# Phase 1 — Dataset Preparation
### SQL Fine-Tuning Project | Spider Dataset → Qwen 1.5B Instruction Format

This notebook:
1. Loads the Spider Text-to-SQL dataset from Hugging Face
2. Formats it into instruction-response pairs
3. Splits into train/validation sets
4. Saves everything to disk


In [1]:
import os
import json
import pandas as pd
from datasets import load_dataset, DatasetDict
from pathlib import Path

print("All imports successful!")


All imports successful!


## Step 1 — Configuration

In [2]:
# ── Project Configuration ──────────────────────────────────────────
CONFIG = {
    "dataset_name"    : "xlangai/spider",   # Spider dataset on HuggingFace
    "model_name"      : "Qwen/Qwen1.5-1.8B-Chat",
    "output_dir"      : "./data",           # Where to save processed data
    "train_split"     : 0.9,               # 90% train, 10% validation
    "max_samples"     : None,              # None = use full dataset
    "seed"            : 42,
}

# Create output directory
Path(CONFIG["output_dir"]).mkdir(parents=True, exist_ok=True)
print("Config ready!")
print(f"Dataset  : {CONFIG['dataset_name']}")
print(f"Model    : {CONFIG['model_name']}")
print(f"Save dir : {CONFIG['output_dir']}")


Config ready!
Dataset  : xlangai/spider
Model    : Qwen/Qwen1.5-1.8B-Chat
Save dir : ./data


## Step 2 — Load Spider Dataset

In [3]:
# Load Spider dataset from Hugging Face
print("Downloading Spider dataset...")
print("This may take a minute on first run...\n")

dataset = load_dataset(CONFIG["dataset_name"])

print("Dataset loaded successfully!")
print(f"\nDataset structure:")
print(dataset)
print(f"\nTrain samples : {len(dataset['train'])}")
print(f"Valid samples  : {len(dataset['validation'])}")


This may take a minute on first run...

Dataset loaded successfully!

Dataset structure:
DatasetDict({
    train: Dataset({
        features: ['db_id', 'query', 'question', 'query_toks', 'query_toks_no_value', 'question_toks'],
        num_rows: 7000
    })
    validation: Dataset({
        features: ['db_id', 'query', 'question', 'query_toks', 'query_toks_no_value', 'question_toks'],
        num_rows: 1034
    })
})

Train samples : 7000
Valid samples  : 1034


## Step 3 — Explore the Data

In [4]:
# Look at a sample entry
sample = dataset['train'][0]
print("Sample entry keys:", list(sample.keys()))
print("\n" + "="*60)
print("QUESTION  :", sample['question'])
print("QUERY     :", sample['query'])
print("DB ID     :", sample['db_id'])
print("="*60)

# Show a few more examples
print("\n--- More examples ---")
for i in [1, 2, 3]:
    s = dataset['train'][i]
    print(f"\nExample {i+1}:")
    print(f"  Q: {s['question']}")
    print(f"  SQL: {s['query']}")


Sample entry keys: ['db_id', 'query', 'question', 'query_toks', 'query_toks_no_value', 'question_toks']

QUESTION  : How many heads of the departments are older than 56 ?
QUERY     : SELECT count(*) FROM head WHERE age  >  56
DB ID     : department_management

--- More examples ---

Example 2:
  Q: List the name, born state and age of the heads of departments ordered by age.
  SQL: SELECT name ,  born_state ,  age FROM head ORDER BY age

Example 3:
  Q: List the creation year, name and budget of each department.
  SQL: SELECT creation ,  name ,  budget_in_billions FROM department

Example 4:
  Q: What are the maximum and minimum budget of the departments?
  SQL: SELECT max(budget_in_billions) ,  min(budget_in_billions) FROM department


## Step 4 — Format into Instruction-Response Pairs

In [5]:
def format_instruction(sample):
    """
    Convert a Spider sample into Qwen chat instruction format.
    
    Input  : Natural language question + database id
    Output : Formatted SQL query
    """
    instruction = f"""You are an expert SQL assistant. Given a natural language question and a database name, write the correct SQL query.

Database: {sample['db_id']}
Question: {sample['question']}

Write only the SQL query, nothing else."""

    response = sample['query'].strip()

    # Qwen chat format
    formatted = f"""<|im_start|>system
You are an expert SQL assistant that converts natural language questions into accurate SQL queries.<|im_end|>
<|im_start|>user
{instruction}<|im_end|>
<|im_start|>assistant
{response}<|im_end|>"""

    return {
        "instruction" : instruction,
        "response"    : response,
        "text"        : formatted,
        "db_id"       : sample['db_id'],
        "question"    : sample['question'],
        "query"       : sample['query'],
    }

# Test the formatter on one sample
test = format_instruction(dataset['train'][0])
print("Formatted sample:")
print("="*60)
print(test['text'])
print("="*60)


Formatted sample:
<|im_start|>system
You are an expert SQL assistant that converts natural language questions into accurate SQL queries.<|im_end|>
<|im_start|>user
You are an expert SQL assistant. Given a natural language question and a database name, write the correct SQL query.

Database: department_management
Question: How many heads of the departments are older than 56 ?

Write only the SQL query, nothing else.<|im_end|>
<|im_start|>assistant
SELECT count(*) FROM head WHERE age  >  56<|im_end|>


In [6]:
# Apply formatting to entire dataset
print("Formatting dataset...")

train_formatted = dataset['train'].map(
    format_instruction,
    remove_columns=dataset['train'].column_names,
    desc="Formatting train set"
)

val_formatted = dataset['validation'].map(
    format_instruction,
    remove_columns=dataset['validation'].column_names,
    desc="Formatting validation set"
)

print(f"\nFormatting complete!")
print(f"Train samples     : {len(train_formatted)}")
print(f"Validation samples: {len(val_formatted)}")


Formatting dataset...

Formatting complete!
Train samples     : 7000
Validation samples: 1034


## Step 5 — Dataset Statistics

In [7]:
# Analyze text lengths
train_lengths = [len(x['text'].split()) for x in train_formatted]
val_lengths   = [len(x['text'].split()) for x in val_formatted]

print("=== Dataset Statistics ===")
print(f"\nTrain set:")
print(f"  Total samples : {len(train_formatted)}")
print(f"  Avg length    : {sum(train_lengths)/len(train_lengths):.0f} tokens")
print(f"  Max length    : {max(train_lengths)} tokens")
print(f"  Min length    : {min(train_lengths)} tokens")

print(f"\nValidation set:")
print(f"  Total samples : {len(val_formatted)}")
print(f"  Avg length    : {sum(val_lengths)/len(val_lengths):.0f} tokens")
print(f"  Max length    : {max(val_lengths)} tokens")
print(f"  Min length    : {min(val_lengths)} tokens")

# Show unique databases
unique_dbs = set(train_formatted['db_id'])
print(f"\nUnique databases in train: {len(unique_dbs)}")
print(f"Sample DBs: {list(unique_dbs)[:8]}")


=== Dataset Statistics ===

Train set:
  Total samples : 7000
  Avg length    : 77 tokens
  Max length    : 152 tokens
  Min length    : 56 tokens

Validation set:
  Total samples : 1034
  Avg length    : 76 tokens
  Max length    : 128 tokens
  Min length    : 56 tokens

Unique databases in train: 140
Sample DBs: ['insurance_fnol', 'architecture', 'coffee_shop', 'flight_1', 'ship_1', 'tracking_share_transactions', 'school_player', 'scientist_1']


## Step 6 — Save Processed Dataset

In [8]:
# Save as HuggingFace DatasetDict
final_dataset = DatasetDict({
    "train"     : train_formatted,
    "validation": val_formatted,
})

# Save to disk
save_path = CONFIG["output_dir"] + "/spider_formatted"
final_dataset.save_to_disk(save_path)
print(f"Dataset saved to: {save_path}")

# Also save as JSON for easy inspection
train_formatted.to_json(CONFIG["output_dir"] + "/train.json")
val_formatted.to_json(CONFIG["output_dir"] + "/validation.json")
print(f"JSON files saved to: {CONFIG['output_dir']}/")

# Save config
with open(CONFIG["output_dir"] + "/dataset_config.json", "w") as f:
    json.dump({
        "dataset_name"       : CONFIG["dataset_name"],
        "train_samples"      : len(train_formatted),
        "validation_samples" : len(val_formatted),
        "format"             : "qwen_chat",
        "model_target"       : CONFIG["model_name"],
    }, f, indent=2)

print("\n=== All files saved! ===")
print(f"  {save_path}/")
print(f"  {CONFIG['output_dir']}/train.json")
print(f"  {CONFIG['output_dir']}/validation.json")
print(f"  {CONFIG['output_dir']}/dataset_config.json")


Saving the dataset (0/1 shards):   0%|          | 0/7000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1034 [00:00<?, ? examples/s]

Dataset saved to: ./data/spider_formatted


Creating json from Arrow format:   0%|          | 0/7 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

JSON files saved to: ./data/

=== All files saved! ===
  ./data/spider_formatted/
  ./data/train.json
  ./data/validation.json
  ./data/dataset_config.json


## Step 7 — Verify Saved Dataset

In [9]:
from datasets import load_from_disk

# Reload and verify
loaded = load_from_disk(CONFIG["output_dir"] + "/spider_formatted")

print("=== Verification ===")
print(f"Train samples     : {len(loaded['train'])}")
print(f"Validation samples: {len(loaded['validation'])}")
print(f"Columns           : {loaded['train'].column_names}")

print("\n--- Sample from saved dataset ---")
sample = loaded['train'][42]
print(f"Question : {sample['question']}")
print(f"Query    : {sample['query']}")
print(f"DB       : {sample['db_id']}")
print("\nPhase 1 Complete! Ready for Phase 2 - Baseline Evaluation")


=== Verification ===
Train samples     : 7000
Validation samples: 1034
Columns           : ['db_id', 'query', 'question', 'instruction', 'response', 'text']

--- Sample from saved dataset ---
Question : Please show the different statuses of cities and the average population of cities with each status.
Query    : SELECT Status ,  avg(Population) FROM city GROUP BY Status
DB       : farm

Phase 1 Complete! Ready for Phase 2 - Baseline Evaluation
